In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


import os, shutil
# from accelerate import notebook_launcher


WORK_DIR = "/kaggle/working/medlens"
SRC_DIR = "/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14"

if not os.path.exists(WORK_DIR):
    shutil.copytree(SRC_DIR, WORK_DIR)

os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/hf_cache"
os.environ["TMPDIR"] = "/kaggle/working/tmp"
os.makedirs("/kaggle/working/tmp", exist_ok=True)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_train.jsonl
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/eval/state.json
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/eval/dataset_info.json
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/eval/data-00000-of-00001.arrow
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/train/state.json
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/train/dataset_info.json
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/train/data-00000-of-00002.arrow
/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_tokenized/train/data-00001-of-00002.arrow


In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    # Local installation
    !pip install unsloth
else:
    # Colab installation
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip uninstall huggingface_hub -y

!pip install huggingface_hub
!pip install -U diffusers transformers
!pip install deepspeed

import torch; torch._dynamo.config.cache_size_limit = 64;  # Or your desired limit

In [28]:
%%writefile ds_config.json

{
  "fp16": {
    "enabled": true,
    "loss_scale": 0,
    "loss_scale_window": 1000,
    "initial_scale_power": 16,
    "hysteresis": 2,
    "min_loss_scale": 1
  },
  "zero_optimization": {
    "stage": 2,
    "allgather_partitions": true,
    "allgather_bucket_size": 2e8,
    "overlap_comm": true,
    "reduce_scatter": true,
    "reduce_bucket_size": 2e8,
    "contiguous_gradients": true
  },
  "gradient_accumulation_steps": "auto",
  "gradient_clipping": "auto",
  "train_batch_size": "auto",
  "train_micro_batch_size_per_gpu": "auto",
  "wall_clock_breakdown": false
}

Overwriting ds_config.json


In [38]:
%%writefile hf_train_script.py

"""
train_script.py — Gemma 4 E2B LoRA fine-tuning with HuggingFace + DeepSpeed ZeRO-2
Replaces Unsloth for multi-GPU (DDP) compatibility on Kaggle T4x2.

Launch command:
  accelerate launch --multi_gpu --num_processes=2 \
    --use_deepspeed --deepspeed_config_file ds_config.json \
    train_script.py
"""

import os
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model


def mask_labels_on_instruction(
    dataset,
    tokenizer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
):
    """
    Recreates Unsloth's `train_on_responses_only` logic:
    Masks (sets to -100) all tokens that are NOT part of the model's response,
    so the loss is only computed on the assistant turns.
    """
    # Encode the delimiter strings (without special tokens)
    response_token_ids = tokenizer.encode(response_part, add_special_tokens=False)
    instruction_token_ids = tokenizer.encode(instruction_part, add_special_tokens=False)

    print(f"Instruction token IDs: {instruction_token_ids}  ->  '{instruction_part.strip()}'")
    print(f"Response token IDs:    {response_token_ids}  ->  '{response_part.strip()}'")

    def _find_all_occurrences(input_ids, pattern):
        """Find all starting indices of `pattern` in `input_ids`."""
        positions = []
        for i in range(len(input_ids) - len(pattern) + 1):
            if input_ids[i : i + len(pattern)] == pattern:
                positions.append(i)
        return positions

    def _create_labels(example):
        input_ids = example["input_ids"]
        labels = list(input_ids)  # start with a copy

        # Find all instruction and response boundaries
        instruction_positions = _find_all_occurrences(input_ids, instruction_token_ids)
        response_positions = _find_all_occurrences(input_ids, response_token_ids)

        if not response_positions:
            # No response found — mask everything (skip this sample during loss)
            example["labels"] = [-100] * len(input_ids)
            return example

        # Build a mask: default to -100 (masked), only unmask response regions
        labels = [-100] * len(input_ids)

        for resp_start in response_positions:
            # The response content starts AFTER the response delimiter tokens
            content_start = resp_start + len(response_token_ids)

            # Find the next instruction start after this response (= end of response)
            content_end = len(input_ids)  # default: until the end
            for inst_start in instruction_positions:
                if inst_start > resp_start:
                    content_end = inst_start
                    break

            # Also check for EOS / padding
            for idx in range(content_start, content_end):
                if input_ids[idx] == tokenizer.eos_token_id or input_ids[idx] == tokenizer.pad_token_id:
                    content_end = idx + 1  # include the EOS token in the loss
                    break

            # Unmask the response tokens
            for idx in range(content_start, content_end):
                labels[idx] = input_ids[idx]

        example["labels"] = labels
        return example

    print("Masking instruction tokens (computing labels)...")
    dataset = dataset.map(_create_labels, num_proc=4, desc="Creating labels")
    return dataset


def main():
    # -------------------------------------------------------------------------
    # Configuration
    # -------------------------------------------------------------------------
    MODEL_NAME = "google/gemma-4-E2B-it"  # use the HF model ID directly
    # MAX_SEQ_LENGTH = 2048
    # OUTPUT_DIR = "outputs"
    # LORA_R = 16           # reduced from 32 — less overfitting risk on 800K samples
    # LORA_ALPHA = 32       # alpha=2*r is a good default

    MAX_SEQ_LENGTH = 512
    TRUNC_LEN = 512
    LORA_R = 8            # 16 → 8 (fewer trainable params, faster)
    LORA_ALPHA = 16       # keep 2x ratio
    OUTPUT_DIR = "outputs"
    
    # -------------------------------------------------------------------------
    # Load tokenizer
    # -------------------------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        model_max_length=MAX_SEQ_LENGTH,
        padding_side="right",
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # -------------------------------------------------------------------------
    # Load pre-tokenized datasets
    # -------------------------------------------------------------------------
    train_dataset = load_from_disk("/kaggle/working/medlens/medlens_tokenized/train")
    eval_dataset = load_from_disk("/kaggle/working/medlens/medlens_tokenized/eval")

    # ---- Truncate pre-tokenized sequences to 1024 ----
    TRUNC_LEN = 1024
    
    def truncate(example):
        example["input_ids"] = example["input_ids"][:TRUNC_LEN]
        example["attention_mask"] = example["attention_mask"][:TRUNC_LEN]
        return example
    
    train_dataset = train_dataset.map(truncate, num_proc=4, desc="Truncating train")
    eval_dataset = eval_dataset.map(truncate, num_proc=4, desc="Truncating eval")
    
    # ---- Subsample: 800K is overkill for 1 epoch on T4s ----
    train_dataset = train_dataset.shuffle(seed=42).select(range(200_000))
    eval_dataset = eval_dataset.shuffle(seed=42).select(range(1_000))

    # -------------------------------------------------------------------------
    # Create labels with response-only masking
    # -------------------------------------------------------------------------
    train_dataset = mask_labels_on_instruction(
        train_dataset, tokenizer,
        instruction_part="<|turn>user\n",
        response_part="<|turn>model\n",
    )
    eval_dataset = mask_labels_on_instruction(
        eval_dataset, tokenizer,
        instruction_part="<|turn>user\n",
        response_part="<|turn>model\n",
    )

    # Sanity check: print a masked example
    sample = train_dataset[100]
    decoded_labels = tokenizer.decode(
        [tokenizer.pad_token_id if x == -100 else x for x in sample["labels"]]
    ).replace(tokenizer.pad_token, " ")
    print(f"\n--- Sanity check (labels for sample 100) ---\n{decoded_labels[:500]}\n")

    # -------------------------------------------------------------------------
    # Load model in 4-bit
    # -------------------------------------------------------------------------
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        dtype=torch.float16,
        device_map={"": int(os.environ.get("LOCAL_RANK", 0))},
        trust_remote_code=True,
    )

    # Manually prepare for training (skip prepare_model_for_kbit_training — it OOMs
    # by casting all params to fp32). We just need to:
    # 1. Freeze base model params
    # 2. Enable gradient checkpointing
    # 3. Disable cache
    model.config.use_cache = False
    model.enable_input_require_grads()  # needed for gradient checkpointing with LoRA
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    # -------------------------------------------------------------------------
    # Attach LoRA adapters
    # -------------------------------------------------------------------------
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,    # light dropout to reduce overfitting
        bias="none",
        task_type="CAUSAL_LM",
        # "all-linear" lets PEFT handle Gemma4ClippableLinear by wrapping
        # the nested nn.Linear inside it, instead of naming modules explicitly
        target_modules="all-linear",
        # Exclude vision/audio encoder layers — text-only fine-tuning
        exclude_modules=["vision_tower", "multi_modal_projector", "audio_tower"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # -------------------------------------------------------------------------
    # Training arguments
    # -------------------------------------------------------------------------
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,        # 1 → 2 (you have VRAM headroom)
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,       # effective batch = 2*2*16 = 64
        num_train_epochs=1,
        learning_rate=3e-5,                   # slightly higher LR for larger batch
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=100,
        eval_strategy="steps",
        eval_steps=1000,                      # less frequent eval
        save_strategy="steps",
        save_steps=2000,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=3407,
        fp16=True,
        bf16=False,
        optim="adamw_8bit",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        ddp_find_unused_parameters=False,
        dataloader_num_workers=4,             # 2 → 4
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        deepspeed="ds_config.json",
    )

    # -------------------------------------------------------------------------
    # Data collator
    # -------------------------------------------------------------------------
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        pad_to_multiple_of=8,
    )

    # -------------------------------------------------------------------------
    # Trainer
    # -------------------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        # tokenizer=tokenizer,
        processing_class=tokenizer
    )

    # -------------------------------------------------------------------------
    # Train!
    # -------------------------------------------------------------------------
    trainer.train()

    # -------------------------------------------------------------------------
    # Save LoRA adapters only
    # -------------------------------------------------------------------------
    trainer.save_model(os.path.join(OUTPUT_DIR, "lora_adapters"))
    tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "lora_adapters"))
    print(f"\nLoRA adapters saved to {OUTPUT_DIR}/lora_adapters")


if __name__ == "__main__":
    main()

Overwriting hf_train_script.py


In [ ]:
!accelerate launch --use_deepspeed --deepspeed_config_file ds_config.json \
    --num_processes=2 \
    hf_train_script.py

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
W0415 12:19:32.692000 4079 torch/distributed/run.py:852] 
W0415 12:19:32.692000 4079 torch/distributed/run.py:852] *****************************************
W0415 12:19:32.692000 4079 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0415 12:19:32.692000 4079 torch/distributed/run.py:852] *****************************************
Instruction token 